# Week 44, Day 5: Beyond one number: distributional and model-based learning

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w044/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Nineteen learners, one distribution

Update all 19 values at once on each of 40,000 samples: V += 2 alpha (tau where the error is positive, 1 - tau where negative) times the error. Average V over the second half. Then compute the true expectiles by bisection on a large sample.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2020)

def sample(n):
    return np.where(r.random(n) < 0.3, r.normal(1, 0.3, n), r.normal(6, 0.8, n))

taus = np.linspace(0.05, 0.95, 19); alpha = 0.01
V = np.full(19, 3.0); acc = np.zeros(19); n_acc = 0
for i, x in enumerate(sample(40000)):
    d = x - V
    ...
    if i >= 20000:
        acc += V; n_acc += 1
V_avg = acc / n_acc

xs = sample(400000)
def expectile(tau):
    lo, hi = xs.min(), xs.max()
    for _ in range(50):
        m = (lo + hi) / 2
        g = tau * np.mean(np.maximum(xs - m, 0)) - (1 - tau) * np.mean(np.maximum(m - xs, 0))
        lo, hi = (m, hi) if g > 0 else (lo, m)
    return m
E_true = np.array([expectile(t) for t in taus])
print("learned:", np.round(V_avg, 2)); print("true:   ", np.round(E_true, 2))
plt.hist(xs, 80, density=True, alpha=0.4); plt.plot(V_avg, np.full(19, 0.02), "k|", ms=20); plt.xlabel("reward"); plt.show()

In [ ]:
import numpy as np
def test_match():
    "each learner's averaged value is within 0.05 of the true expectile"
    assert np.abs(V_avg - E_true).max() < 0.05
def test_spread():
    "the population spans more than 4 reward units and its middle learner sits near the mean"
    assert V_avg[-1] - V_avg[0] > 4 and abs(V_avg[9] - xs.mean()) < 0.1

test_match()
test_spread()
print("All checks passed")

### Exercise: The two-step task

Complete the first-stage valuation: the model-based value of each first-stage action is the transition matrix times the best second-stage value in each state; the agent uses w times the model-based value plus (1 - w) times the model-free value. Then compute stay probabilities by previous reward and transition.

In [ ]:
def two_step(w, T=5000, alpha=0.5, beta=5.0, seed=2011):
    g = np.random.default_rng(seed)
    Q1_mf = np.zeros(2); Q2 = np.full((2, 2), 0.5); P = g.uniform(0.25, 0.75, (2, 2))
    trans = np.array([[0.7, 0.3], [0.3, 0.7]])
    rec = []
    for t in range(T):
        Q1_mb = ...
        Q1 = ...
        a = int(g.random() < 1 / (1 + np.exp(-beta * (Q1[1] - Q1[0]))))
        common = g.random() < 0.7; s = a if common else 1 - a
        b = int(g.random() < 1 / (1 + np.exp(-beta * (Q2[s, 1] - Q2[s, 0]))))
        rew = float(g.random() < P[s, b])
        Q1_mf[a] += alpha * (Q2[s, b] - Q1_mf[a])          # first-stage TD update
        d2 = rew - Q2[s, b]; Q2[s, b] += alpha * d2; Q1_mf[a] += alpha * d2   # second stage, credit passed back
        P = np.clip(P + 0.025 * g.normal(size=(2, 2)), 0.25, 0.75)
        rec.append((a, common, rew))
    rec = np.array(rec)
    stay_ = rec[1:, 0] == rec[:-1, 0]; c = rec[:-1, 1].astype(bool); rw_ = rec[:-1, 2].astype(bool)
    return {k: stay_[m].mean() for k, m in [("rew common", rw_ & c), ("rew rare", rw_ & ~c), ("unrew common", ~rw_ & c), ("unrew rare", ~rw_ & ~c)]}

stay = {w: two_step(w) for w in [0.0, 1.0, 0.5]}
for w, s in stay.items():
    print(f"w = {w}: " + ", ".join(f"{k} {v:.2f}" for k, v in s.items()))
fig, ax = plt.subplots(1, 3, figsize=(9, 3), sharey=True)
for a_, (w, s) in zip(ax, stay.items()):
    a_.bar([0, 1, 3, 4], [s["rew common"], s["rew rare"], s["unrew common"], s["unrew rare"]], color=["C0", "C1", "C0", "C1"]); a_.set_title(f"w = {w}"); a_.set_xticks([0.5, 3.5], ["rewarded", "unrewarded"])
ax[0].set_ylabel("stay probability"); ax[0].set_ylim(0.3, 1); plt.show()

In [ ]:
import numpy as np
def test_mf():
    "model-free: staying depends on reward, not on transition"
    s = stay[0.0]
    assert s["rew common"] - s["unrew common"] > 0.2 and abs(s["rew common"] - s["rew rare"]) < 0.05
def test_mb():
    "model-based: reward and transition interact (staying after rewarded common and unrewarded rare)"
    s = stay[1.0]
    assert s["rew common"] - s["rew rare"] > 0.1 and s["unrew rare"] - s["unrew common"] > 0.1

test_mf()
test_mb()
print("All checks passed")